In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

import pymupdf
import pymupdf4llm

from sentence_transformers import SentenceTransformer

from pymilvus import (
    MilvusClient, DataType, Function, FunctionType
)

from tqdm import tqdm
from pathlib import Path

c:\AMORE\Development\Python Projects\yet-another-RAG-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def pdf_preprocess(file_path):
    doc = pymupdf.open(file_path)

    pages = []

    for page_num in range(len(doc)):
        md_text = pymupdf4llm.to_markdown(
            doc,
            pages=[page_num]
        )

        pages.append({
            "page": page_num,
            "text": md_text
        })

    return pages, Path(file_path).name

In [3]:
def chunk_text(pages):
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header 1"),
            ("##", "Header 2")
        ],
        strip_headers=False
    )

    chunks_with_metadata = []

    for page_data in pages:
        page_chunks = splitter.split_text(page_data["text"])

        for chunk in page_chunks:
            chunks_with_metadata.append({
                "text": chunk.page_content,
                "page_number": page_data["page"]
            })

    return chunks_with_metadata

In [4]:
def emb_text(text, embedding_model):
    emb = embedding_model.encode(text,normalize_embeddings=True)
    return emb.tolist()

In [ ]:
def create_milvus_collection(milvus_client, embedding_dim, collection_name, metric_type, max_length):
    if milvus_client.has_collection(collection_name):
        milvus_client.drop_collection(collection_name)
        
    schema = milvus_client.create_schema(auto_id=False)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, description="ID")
    schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=max_length, enable_analyzer=True, description="СЫРОЙ ТЕКСТ")
    schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=embedding_dim, description="ЭМБЕДДИНГ")
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR, description="РАЗРЕЖЕННЫЙ B25")
    schema.add_field(field_name="document_name", datatype=DataType.VARCHAR, max_length=300, description="НАЗВАНИЕ ДОКУМЕНТА")
    schema.add_field(field_name="page_number", datatype=DataType.INT64, description="СТРАНИЦА В ДОКУМЕНТЕ")
    
    bm25_function = Function(
        name="text_bm25_emb",
        input_field_names=["text"],
        output_field_names=["sparse"],
        function_type=FunctionType.BM25,
    )
    schema.add_function(bm25_function)
    
    index_params = milvus_client.prepare_index_params()
    index_params.add_index(
        field_name="id",
        index_type="AUTOINDEX"
    )
    index_params.add_index(
        field_name="embedding", 
        index_type="AUTOINDEX",
        metric_type=metric_type
    )
    index_params.add_index(
        field_name="sparse",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="BM25",
        params={
            "inverted_index_algo": "DAAT_MAXSCORE",
            "bm25_k1": 1.2,
            "bm25_b": 0.75
        }
    )

    milvus_client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )
    
    print(f"Коллекция {collection_name} создана")

In [6]:
def vectordb_insert(
    milvus_client,
    chunks,
    collection_name,
    embedding_model,
    filename
):
    data = []

    for i, chunk in enumerate(chunks):
        text = chunk["text"]

        data.append({
            "id": i,
            "text": text,
            "embedding": emb_text(text, embedding_model),
            "document_name": filename,
            "page_number": chunk["page_number"] + 1
        })

    milvus_client.insert(
        collection_name=collection_name,
        data=data
    )

    print(
        f"Сохранено {len(data)} документов "
        f"в коллекцию {collection_name}"
    )

In [7]:
def ingest_pipeline(
    file_path,
    milvus_client,
    collection_name,
    embedding_model_name
):
    md_text, filename = pdf_preprocess(file_path)

    chunks = chunk_text(md_text)

    embedding_model = SentenceTransformer(
        embedding_model_name,
        cache_folder="models_cache"
    )

    print("Модель эмбеддингов загружена")

    vectordb_insert(
        milvus_client,
        chunks,
        collection_name,
        embedding_model,
        filename
    )

    return "Пайплайн прошёл успешно!"

In [9]:
milvus_client = MilvusClient(
    uri="http://localhost:19530"
)

embedding_dim = 384
collection_name = "UZConstitution_RAG"
metric_type = "IP"
embedding_model_name="intfloat/multilingual-e5-small"

In [10]:
create_milvus_collection(milvus_client, embedding_dim, collection_name, metric_type, max_length=8000)

Коллекция UZConstitution_RAG создана


In [ ]:
file_path = r""

In [17]:
ingest_pipeline(
    file_path=file_path,
    milvus_client=milvus_client,
    collection_name=collection_name,
    embedding_model_name=embedding_model_name
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2750.25it/s]


Модель эмбеддингов загружена
Сохранено 34 документов в коллекцию UZConstitution_RAG


'Пайплайн прошёл успешно!'